In [ ]:
!git clone https://github.com/d4-5/NLP4.git
%cd NLP4

In [ ]:
!pip install -r requirements.txt

In [ ]:
%cd NLP4

In [1]:
import stanza
stanza.download('uk')

/home/user/lababd10/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-24 18:46:12 INFO: Downloaded file to /home/user/.cache/stanza/1.11.0/resources/resources.json
2026-04-24 18:46:12 INFO: Downloading default packages for language: uk (Ukrainian) ...
2026-04-24 18:46:46 INFO: Downloaded file to /home/user/.cache/stanza/1.11.0/resources/uk/default.zip
2026-04-24 18:46:48 INFO: Finished downloading models and saved to /home/user/.cache/stanza/1.11.0/resources


[['zip', 'default.zip']]

In [4]:
from pathlib import Path
import sys

BASE_DIR = Path('..').resolve()
sys.path.append(str(BASE_DIR))

EVAL_PATH = BASE_DIR / 'data' / 'sample' / 'lab10_ner_eval.jsonl'
BASELINE_OUT = BASE_DIR / 'data' / 'sample' / 'lab10_baseline_predictions.jsonl'
HYBRID_OUT = BASE_DIR / 'data' / 'sample' / 'lab10_hybrid_predictions.jsonl'

print('BASE_DIR =', BASE_DIR)
print('EVAL_PATH =', EVAL_PATH)

BASE_DIR = /home/user/lababd10
EVAL_PATH = /home/user/lababd10/data/sample/lab10_ner_eval.jsonl


In [5]:
from src.ner_pipeline import read_jsonl

eval_records = read_jsonl(EVAL_PATH)
len(eval_records), eval_records[0]['text_id']

(20, 'text_9448')

In [6]:
for row in eval_records[:3]:
    print('TEXT ID:', row['text_id'])
    print('COMMENT:', row['comment'])
    print('EXPECTED:', [(ent['text'], ent['label']) for ent in row['expected_entities'] if ent['label'] in {'PERS','ORG','DATE','MON','LOC'}])
    print('-' * 80)

TEXT ID: text_9448
COMMENT: Короткий кейс з PERSON, ORG, DATE, MON і кількісною сутністю; зручний для baseline demo.
EXPECTED: [('Романків', 'LOC'), ('Юрій Атаманюк', 'PERS'), ('Вільногірський гірничо-металургійний комбінат', 'ORG'), ("ДП «Об'єднана гірничо-хімічна компанія»", 'ORG'), ('23 червня', 'DATE'), ('64,68 млн грн', 'MON')]
--------------------------------------------------------------------------------
TEXT ID: text_10688
COMMENT: Показує корпоративну назву з ТОВ, відносну дату та суму у грн.
EXPECTED: [('ТОВ «Науково-практичний центр реконструкції історичного середовища»', 'ORG'), ('липні цього року', 'DATE'), ('5 тис грн.', 'MON'), ('Костянтин Леонідович Роєнко', 'PERS'), ('Буча', 'LOC'), ('Київської області', 'LOC')]
--------------------------------------------------------------------------------
TEXT ID: text_9744
COMMENT: Добрий приклад довших ORG-спанів і кількох PERSON в одному документі.
EXPECTED: [('Аеропорт «Бориспіль»', 'ORG'), ('2009 року', 'DATE'), ('ТОВ «Еко-Сер

In [7]:
from src.ner_pipeline import load_stanza_pipeline

pipeline = load_stanza_pipeline(lang='uk', use_gpu=False)
pipeline

2026-04-24 18:47:46 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-04-24 18:47:46 INFO: Downloaded file to /home/user/.cache/stanza/1.11.0/resources/resources.json
2026-04-24 18:47:46 WARNING: Language uk package default expects mwt, which has been added
2026-04-24 18:47:46 INFO: Loading these models for language: uk (Ukrainian):
| Processor | Package |
-----------------------
| tokenize  | iu      |
| mwt       | iu      |
| ner       | languk  |

2026-04-24 18:47:46 INFO: Using device: cpu
2026-04-24 18:47:46 INFO: Loading: tokenize
2026-04-24 18:47:48 INFO: Loading: mwt
2026-04-24 18:47:48 INFO: Loading: ner
2026-04-24 18:47:51 INFO: Done loading processors!


In [8]:
from src.ner_pipeline import run_baseline, write_jsonl

baseline_records = run_baseline(eval_records, pipeline)
write_jsonl(BASELINE_OUT, baseline_records)
print('saved to', BASELINE_OUT)

saved to /home/user/lababd10/data/sample/lab10_baseline_predictions.jsonl


In [9]:
for row in baseline_records[:12]:
    print('TEXT ID:', row['text_id'])
    print('TEXT:', row['text'][:220], '...')
    print('PREDICTED:', [(ent['text'], ent['label']) for ent in row['predicted_entities'] if ent['label'] in {'PERS','ORG','DATE','MON','LOC'}])
    print('EXPECTED:', [(ent['text'], ent['label']) for ent in row['expected_entities'] if ent['label'] in {'PERS','ORG','DATE','MON','LOC'}])
    print('=' * 100)

TEXT ID: text_9448
TEXT: Крім того, як раніше повідомляв проект "Гарна Хата", у цьому ж селі Романків за час служби збудував маєток на 600 кв. м. екс-заступник голови податкової міліції Юрій Атаманюк "Вільногірський гірничо-металургійний комбіна ...
PREDICTED: [('Гарна Хата"', 'ORG'), ('Романків', 'LOC'), ('Юрій Атаманюк', 'PERS'), ('Вільногірський гірничо-металургійний комбінат "ДП "Об\'єднана гірничо-хімічна компанія" 23 червня', 'ORG')]
EXPECTED: [('Романків', 'LOC'), ('Юрій Атаманюк', 'PERS'), ('Вільногірський гірничо-металургійний комбінат', 'ORG'), ("ДП «Об'єднана гірничо-хімічна компанія»", 'ORG'), ('23 червня', 'DATE'), ('64,68 млн грн', 'MON')]
TEXT ID: text_10688
TEXT: І всі три учасники торгів зробили заявки лише на піввідсотка менші від очікуваної вартості. ТОВ "Науково-практичний центр реконструкції історичного середовища" засновано у липні цього року, за тиждень до оголошення тенде ...
PREDICTED: [('Науково-практичний центр реконструкції історичного середовища"', 'ORG'), 

In [10]:
from src.ner_rules import money_rule, date_rule, legal_form_org_rule

sample_text = eval_records[0]['text']
print('money_rule:', money_rule(sample_text))
print('date_rule:', date_rule(sample_text))
print('legal_form_org_rule:', legal_form_org_rule(sample_text))

money_rule: [{'text': '64,68 млн грн.', 'label': 'MON', 'start_char': 313, 'end_char': 327, 'source': 'rule:money_regex_v1'}]
date_rule: [{'text': '23 червня', 'label': 'DATE', 'start_char': 263, 'end_char': 272, 'source': 'rule:date_regex_v1'}]
legal_form_org_rule: [{'text': 'ДП "Об\'єднана гірничо-хімічна компанія"', 'label': 'ORG', 'start_char': 223, 'end_char': 262, 'source': 'rule:org_legal_form_v1'}]


In [11]:
from src.ner_pipeline import run_hybrid

hybrid_records = run_hybrid(eval_records, pipeline)
write_jsonl(HYBRID_OUT, hybrid_records)
print('saved to', HYBRID_OUT)

saved to /home/user/lababd10/data/sample/lab10_hybrid_predictions.jsonl


In [12]:
from src.ner_eval import compare_runs

comparison = compare_runs(baseline_records, hybrid_records)
comparison

{'baseline_metrics': {'DATE': {'gold': 24,
   'predicted': 0,
   'correct': 0,
   'missed': 24,
   'false_positive': 0,
   'rough_precision': 0.0,
   'rough_recall': 0.0},
  'LOC': {'gold': 30,
   'predicted': 34,
   'correct': 6,
   'missed': 24,
   'false_positive': 28,
   'rough_precision': 0.1765,
   'rough_recall': 0.2},
  'MON': {'gold': 23,
   'predicted': 0,
   'correct': 0,
   'missed': 23,
   'false_positive': 0,
   'rough_precision': 0.0,
   'rough_recall': 0.0},
  'ORG': {'gold': 78,
   'predicted': 77,
   'correct': 8,
   'missed': 70,
   'false_positive': 69,
   'rough_precision': 0.1039,
   'rough_recall': 0.1026},
  'PERS': {'gold': 43,
   'predicted': 43,
   'correct': 10,
   'missed': 33,
   'false_positive': 33,
   'rough_precision': 0.2326,
   'rough_recall': 0.2326}},
 'hybrid_metrics': {'DATE': {'gold': 24,
   'predicted': 20,
   'correct': 4,
   'missed': 20,
   'false_positive': 16,
   'rough_precision': 0.2,
   'rough_recall': 0.1667},
  'LOC': {'gold': 30,
   

In [13]:
from src.ner_eval import collect_errors, error_summary

errors = collect_errors(hybrid_records)
print('Total errors:', len(errors))
print('Summary:', error_summary(errors))

for error in errors[:15]:
    expected = error['expected_entity']
    predicted = error['predicted_entity']
    print('TEXT ID:', error['text_id'])
    print('CATEGORY:', error['category'])
    print('EXPECTED:', None if expected is None else (expected['text'], expected['label']))
    print('PREDICTED:', None if predicted is None else (predicted['text'], predicted['label']))
    print('EXPLANATION:', error['explanation'])
    print('-' * 100)

Total errors: 183
Summary: {'boundary error': 146, 'false positive': 12, 'normalization issue': 10, 'type error': 9, 'ambiguous case': 3, 'tokenization / normalization issue': 2, 'missed domain entity': 1}
TEXT ID: text_9448
CATEGORY: boundary error
EXPECTED: ('Романків', 'LOC')
PREDICTED: ('Романків', 'LOC')
EXPLANATION: Модель знайшла близький span, але межі сутності не збіглися з gold-розміткою.
----------------------------------------------------------------------------------------------------
TEXT ID: text_9448
CATEGORY: boundary error
EXPECTED: ('Юрій Атаманюк', 'PERS')
PREDICTED: ('Юрій Атаманюк', 'PERS')
EXPLANATION: Модель знайшла близький span, але межі сутності не збіглися з gold-розміткою.
----------------------------------------------------------------------------------------------------
TEXT ID: text_9448
CATEGORY: boundary error
EXPECTED: ('Вільногірський гірничо-металургійний комбінат', 'ORG')
PREDICTED: ('"Вільногірський гірничо-металургійний комбінат "ДП "Об\'єднана г